# RAG Pipeline & Sandbox BANASPATI (Orang 2)

Notebook ini mengimplementasikan sistem retrieval dan LLM generation dengan model open-weight <9B menggunakan HuggingFace API.

In [2]:
import os, warnings
warnings.filterwarnings('ignore')

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import ipywidgets as widgets
from IPython.display import display, Markdown


# API Key Gemini dari Orang 2, kalau habis ntar siapapun buat api lagi
os.environ["GEMINI_API_KEY"] = "MASUKKAN_GEMINI_API_KEY_ANDA_DISINI"


## 1. Load Vector Database (ChromaDB)

In [3]:
embed_model = HuggingFaceEmbeddings(model_name='paraphrase-multilingual-MiniLM-L12-v2')
persist_dir = '../artifacts/chroma' if os.path.basename(os.getcwd()) == 'notebooks' else 'artifacts/chroma'
vectorstore = Chroma(persist_directory=persist_dir, collection_name='banaspati', embedding_function=embed_model)
retriever = vectorstore.as_retriever(search_kwargs={'k': 25})


## 2. Setup Generator LLM (<9B Parameter)

In [4]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.1,
)


## 3. Prompt Engineering (Anti-Halusinasi)

In [5]:
template = """Anda adalah BANASPATI (Bubur Panas Personal Assistant), asisten akademik yang ahli.
Tugas Anda adalah menjawab pertanyaan pengguna HANYA BERDASARKAN KONTEKS DOKUMEN yang diberikan di bawah ini.

ATURAN KETAT:
1. Baca dan pahami SELURUH konteks dokumen dengan teliti sebelum menjawab.
2. Jika jawabannya ADA di dalam konteks (meskipun tersebar di beberapa dokumen), rangkum dan jawab dengan lengkap.
3. Jika informasi BENAR-BENAR tidak ada di dalam konteks, jawab HANYA dengan: "informasi tidak ditemukan"
4. Jangan pernah mengarang jawaban atau menggunakan pengetahuan dari luar dokumen.

=== KONTEKS DOKUMEN ===
{context}
=== AKHIR KONTEKS ===

Pertanyaan: {question}

Jawaban:"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])


## 4. Fungsi Utama dan Sandbox UI

In [6]:
def banaspati_answer(question):
    # Hybrid retrieval: semantic search + keyword-boosted reranking
    raw_docs = vectorstore.similarity_search(question, k=40)
    
    # Simple keyword reranking: boost docs that contain question keywords
    question_words = set(question.lower().split())
    stopwords = {'yang', 'di', 'dan', 'dari', 'ini', 'itu', 'ada', 'dalam', 'untuk', 'pada', 'ke', 'dengan', 'adalah', 'apa', 'atau', 'sebutkan', 'jelaskan', 'beberapa', 'bagaimana', 'berapa', 'apakah', 'serta', 'akan', 'oleh', 'tidak', 'jika', 'maka', 'saat', 'bisa', 'dapat', 'telah', 'sudah', 'harus', 'juga', 'tersebut', 'dokumen'}
    keywords = question_words - stopwords
    
    def score_doc(doc):
        text_lower = doc.page_content.lower()
        # Penalize very short docs (likely TOC/headers)
        length_penalty = 0 if len(doc.page_content) > 200 else -5
        # Count keyword hits
        keyword_hits = sum(1 for kw in keywords if kw in text_lower)
        # Bonus for docs with actual data (numbers, codes like ET234xxx)
        has_course_code = 1 if 'ET234' in doc.page_content or 'UG234' in doc.page_content or 'SM234' in doc.page_content or 'SF234' in doc.page_content or 'EE234' in doc.page_content else 0
        return keyword_hits + has_course_code * 2 + length_penalty
    
    # Sort by combined score (descending), keep top 10
    scored = sorted(raw_docs, key=score_doc, reverse=True)
    docs = scored[:10]
    
    context_text = "\n\n".join([f"[Sumber: {d.metadata.get('source_file', 'Unknown')}, Hal: {d.metadata.get('page', 'Unknown')}]\n{d.page_content}" for d in docs])
    
    print("="*50)
    print("DOKUMEN REFERENSI YANG DI-RETRIEVE:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. {doc.metadata.get('source_file', '')} (Hal {doc.metadata.get('page', '')})")
        print(doc.page_content[:200] + "...")
    print("="*50 + "\n")
    
    final_prompt = prompt.format(context=context_text, question=question)
    answer = llm.invoke(final_prompt)
    return answer.content

# Sandbox Widget
out = widgets.Output()
text_input = widgets.Text(description="Tanya:", layout=widgets.Layout(width="80%"))
button = widgets.Button(description="Kirim", button_style="success")

def on_submit(_):
    button.disabled = True
    button.description = "Loading..."
    try:
        with out:
            out.clear_output()
            q = text_input.value
            display(Markdown(f"**Pertanyaan:** {q}"))
            ans = banaspati_answer(q)
            display(Markdown(f"**BANASPATI:** {ans}"))
    finally:
        button.disabled = False
        button.description = "Kirim"

button.on_click(on_submit)
display(widgets.HBox([text_input, button]), out)


Output()